# Respuestas del profesor Qwen3-8B en Kaggle (IIC-3694)

El profesor resuelve tareas de `train` para dar respuestas correctas donde el modelo objetivo (Qwen3-1.7B) no las tiene. Después, `build_training_data.py` junta las respuestas de ambos: SFT prioriza las del profesor y DPO arma pares con una correcta y una incorrecta del mismo prompt.

Corre en tres pasos, que se eligen con `PASO` en la configuración:

1. **Prueba:** 9 prompts donde el 1.7B nunca acertó, sobre todo de composición. Mide si el 8B los resuelve y cuánto tarda.
2. **Pendientes:** los 71 prompts sin respuesta correcta.
3. **Todo:** los 180 prompts de `train`, para que todas las trazas SFT vengan del profesor.

Cada paso incluye los anteriores y **continúa lo ya hecho** si le das como entrada la salida de la versión anterior. Qwen3-8B pesa 16 GB en float16: se reparte entre las dos T4 (*GPU T4 x2*). Detalle en `docs/guia_kaggle.md`.

## 1. GPU

In [ ]:
import os
import time

INICIO = time.time()
PRUEBA_SECA = os.environ.get("PRUEBA_SECA") == "1"  # solo para probar el cuaderno sin GPU

if PRUEBA_SECA:
    N_GPU = 2
else:
    import torch

    N_GPU = torch.cuda.device_count()
    for i in range(N_GPU):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {props.name} ({props.total_memory / 2**30:.1f} GiB)")
if N_GPU < 2:
    print("=" * 70)
    print("ATENCION: hacen falta dos GPU. Settings > Accelerator > GPU T4 x2.")
    print("Con una sola T4, el 8B se carga en 8 bits y es mucho mas lento.")
    print("=" * 70)

## 2. Código y resultados anteriores

Busca en `/kaggle/input` el código (debe traer `scripts/pending_prompts.py`), las respuestas del 1.7B (`trazas_qwen17b`) y, si existe, una corrida anterior del 8B (`trazas_qwen8b`) para continuarla.

In [ ]:
import shutil
import sys
import zipfile
from pathlib import Path

ENTRADA = Path(os.environ.get("ENTRADA_KAGGLE", "/kaggle/input"))
SALIDA = Path(os.environ.get("SALIDA_KAGGLE", "/kaggle/working"))
CODIGO = Path(os.environ.get("CODIGO_KAGGLE", "/tmp/inventory_harness"))
RESULTADOS = SALIDA / "results"
RESULTADOS.mkdir(parents=True, exist_ok=True)


def es_codigo_nuevo(raiz):
    return (raiz / "scripts" / "pending_prompts.py").exists() and (raiz / "src" / "inventory_harness").is_dir()


def buscar_codigo():
    for script in sorted(ENTRADA.glob("**/scripts/pending_prompts.py")):
        if es_codigo_nuevo(script.parent.parent):
            return script.parent.parent
    for ruta in sorted(ENTRADA.glob("**/inventory_harness_colab.zip")):
        if ruta.is_file():
            with zipfile.ZipFile(ruta) as archivo:
                if "scripts/pending_prompts.py" in archivo.namelist():
                    return ruta
    return None


origen = buscar_codigo()
if origen is None:
    raise SystemExit("No encontre el codigo nuevo (con scripts/pending_prompts.py) en /kaggle/input.")
shutil.rmtree(CODIGO, ignore_errors=True)
if origen.is_file():
    with zipfile.ZipFile(origen) as archivo:
        archivo.extractall(CODIGO)
else:
    shutil.copytree(origen, CODIGO)
sys.path.insert(0, str(CODIGO / "src"))
print("Codigo desde", origen)


def rollouts_hechos(carpeta):
    total = 0
    for archivo in carpeta.glob("rollouts_*.jsonl"):
        with archivo.open(encoding="utf-8", errors="replace") as f:
            total += sum(1 for linea in f if linea.strip())
    return total


def buscar_corrida(nombre):
    return sorted(m.parent for m in ENTRADA.glob(f"**/{nombre}/manifest.json"))


chico = buscar_corrida("trazas_qwen17b")
if not chico:
    raise SystemExit("No encontre trazas_qwen17b en /kaggle/input. Sube resultados_qwen17b.zip al dataset.")
DIR_CHICO = RESULTADOS / "trazas_qwen17b"
if not DIR_CHICO.exists():
    shutil.copytree(chico[0], DIR_CHICO)
print(f"1.7B: {chico[0]} ({rollouts_hechos(DIR_CHICO)} rollouts)")

DIR_PROFESOR = RESULTADOS / "trazas_qwen8b"
previa = buscar_corrida("trazas_qwen8b")
if DIR_PROFESOR.exists():
    print(f"8B: ya esta en {DIR_PROFESOR} ({rollouts_hechos(DIR_PROFESOR)} rollouts); se continua.")
elif previa:
    shutil.copytree(previa[0], DIR_PROFESOR)
    print(f"8B: copiada desde {previa[0]} ({rollouts_hechos(DIR_PROFESOR)} rollouts); se continua.")
else:
    print("8B: no hay corrida anterior; empieza de cero.")

## 3. Dependencias

In [ ]:
import subprocess

if not PRUEBA_SECA:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "transformers>=4.51", "accelerate"],
        check=True,
    )

## 4. Configuración

**Cambia solo `PASO`.** 1 = prueba con 9 prompts, 2 = los 71 pendientes, 3 = los 180 de `train`. `LIMITE_HORAS` detiene la corrida a tiempo para guardar antes del tope de 12 horas de Kaggle.

In [ ]:
PASO = 1
PRUEBA_N = 9
LIMITE_HORAS = 11.0

# ------------------- No hace falta editar desde aquí -------------------
MODELO = "Qwen/Qwen3-8B"
ETIQUETA = MODELO + "+think"
BATCH_SIZE = 8
MUESTRAS = 2  # igual que las trazas del 1.7B
POR_CELDA = 10
LIMITE = INICIO + LIMITE_HORAS * 3600
SECO = ["--dry-run"] if PRUEBA_SECA else []
DIR_DATOS = RESULTADOS / "datos_derivados_qwen8b"
PENDIENTES = RESULTADOS / "pendientes.json"
PRUEBA = RESULTADOS / "pendientes_prueba.json"
assert PASO in (1, 2, 3), "PASO debe ser 1, 2 o 3"


def correr(*argumentos, limite=None):
    """Ejecuta un script del proyecto, muestra su salida y lo detiene si pasa `limite`."""
    proceso = subprocess.Popen(
        [sys.executable, "-u", *map(str, argumentos)], cwd=CODIGO,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    import threading

    hilo = threading.Thread(target=lambda: [print(l, end="", flush=True) for l in proceso.stdout], daemon=True)
    hilo.start()
    while proceso.poll() is None:
        if limite is not None and time.time() > limite:
            print(f"Se alcanzo LIMITE_HORAS={LIMITE_HORAS}: se detiene para guardar.", flush=True)
            proceso.terminate()
            try:
                proceso.wait(timeout=120)
            except subprocess.TimeoutExpired:
                proceso.kill()
            break
        time.sleep(2 if PRUEBA_SECA else 20)
    proceso.wait()
    hilo.join(timeout=10)
    return proceso.returncode


# Las listas salen solo del 1.7B, para que sean las mismas en cada version.
correr("scripts/pending_prompts.py", DIR_CHICO, "--out", PENDIENTES)
correr("scripts/pending_prompts.py", DIR_CHICO, "--out", PRUEBA, "--limit", PRUEBA_N)

## 5. Correr el profesor

Primero descarga el modelo (16 GB, unos minutos). Luego corre cada paso hasta el elegido; lo ya hecho no se repite. Cada ~25 respuestas se imprime el avance.

In [ ]:
if not PRUEBA_SECA:
    from huggingface_hub import snapshot_download

    snapshot_download(MODELO)

BASE = [
    "scripts/run_pilot.py", "--stage", "trazas", "--models", MODELO, "--thinking", "on",
    "--n-per-cell", POR_CELDA, "--n-samples", MUESTRAS, "--batch-size", BATCH_SIZE,
    "--out", DIR_PROFESOR, *SECO,
]
pasos = [("prueba", ["--only", PRUEBA]), ("pendientes", ["--only", PENDIENTES]), ("todo", [])][:PASO]
for nombre, extra in pasos:
    if time.time() > LIMITE - 600:
        print(f"Sin tiempo para el paso '{nombre}'.")
        break
    print("=" * 20, f"Paso: {nombre}", "=" * 20, flush=True)
    codigo = correr(*BASE, *extra, limite=LIMITE)
    print(f"Paso '{nombre}' termino con codigo {codigo}", flush=True)
print(f"Tiempo total: {(time.time() - INICIO) / 3600:.2f} h")

## 6. ¿El 8B resuelve lo que el 1.7B no?

Para cada prompt pendiente que ya corrió el 8B: si acertó en alguna de sus 2 muestras. Se usa el mismo criterio que para SFT (recompensa 1 y llamadas válidas).

In [ ]:
import json
from collections import Counter, defaultdict

from inventory_harness.reward import InstanceStore
from inventory_harness.training_data import collect

if (DIR_PROFESOR / "manifest.json").exists():
    filas = []
    for archivo in DIR_PROFESOR.glob("rollouts_*.jsonl"):
        for linea in archivo.read_text(encoding="utf-8").splitlines():
            try:
                filas.append(json.loads(linea))
            except ValueError:
                pass
    sys.path.insert(0, str(CODIGO / "scripts"))
    import run_pilot

    store = InstanceStore(run_pilot.build_dataset("train", POR_CELDA, 2026))
    candidatos = collect(filas, store, Counter())
    resuelto = defaultdict(bool)
    for c in candidatos:
        resuelto[(c.instance.id, c.condition)] |= c.correct
    pendientes = json.loads(PENDIENTES.read_text(encoding="utf-8"))["pending"]
    por_celda = defaultdict(lambda: [0, 0])
    for p in pendientes:
        clave = (p["instance_id"], p["condition"])
        if clave in resuelto:
            celda = f"{p['family']}|{p['difficulty']}|{p['condition']}"
            por_celda[celda][0] += resuelto[clave]
            por_celda[celda][1] += 1
    hechos = sum(t for _, t in por_celda.values())
    bien = sum(b for b, _ in por_celda.values())
    print(f"Pendientes que ya corrio el 8B: {hechos}; resueltos: {bien}")
    for celda, (b, t) in sorted(por_celda.items()):
        print(f"  {celda}: {b}/{t}")
    otros = {k for k in resuelto} - {(p["instance_id"], p["condition"]) for p in pendientes}
    if otros:
        print(f"Prompts que el 1.7B ya resolvia: {len(otros)}; el 8B acerto {sum(resuelto[k] for k in otros)}")
    segundos = [json.loads(l)["usage"]["seconds"] for a in DIR_PROFESOR.glob("rollouts_*.jsonl")
                for l in a.read_text(encoding="utf-8").splitlines() if l.strip()]
    if segundos and not PRUEBA_SECA:
        print(f"Tiempo medio por respuesta (suma de usage.seconds): {sum(segundos) / len(segundos):.0f} s")

## 7. Datos SFT/DPO con ambos modelos y zip

In [ ]:
correr("scripts/analyze_results.py", DIR_PROFESOR)
correr("scripts/build_training_data.py", DIR_CHICO, DIR_PROFESOR, "--out", DIR_DATOS, "--teacher", ETIQUETA)

import datetime

ZIP_RESULTADOS = SALIDA / "resultados_qwen8b.zip"
with zipfile.ZipFile(ZIP_RESULTADOS, "w", zipfile.ZIP_DEFLATED) as archivo:
    for carpeta in (DIR_PROFESOR, DIR_DATOS):
        if carpeta.exists():
            for ruta in sorted(carpeta.iterdir()):
                if ruta.is_file() and not ruta.name.endswith(".tmp"):
                    archivo.write(ruta, f"{carpeta.name}/{ruta.name}")
    for ruta in (PENDIENTES, PRUEBA):
        archivo.write(ruta, ruta.name)
print(datetime.datetime.now().isoformat(timespec="seconds"))
print(f"LISTO ({ZIP_RESULTADOS.stat().st_size / 2**20:.1f} MiB): {ZIP_RESULTADOS}")
print("Descargalo desde la pestana Output de esta version.")